# ML-08 — A frozen tree for the refresh-review queue

Lane 2 remains locked. This notebook trains a small decision tree on real March warehouse features and compares it with the frozen Week 4 volume rule on exactly the same client split. It does not select parameters using June. See [the frozen protocol](../capstone_protocol.json) and [capstone pipeline](../scripts/capstone_pipeline.py).

Run from a cloned repo with `work/requirements-capstone.txt` installed and Hugging Face access. The final aggregate receipt is committed; the pipeline rebuilds it from pinned data. No private page text or raw query strings are used.

## 1. Method choice and why

A maximum-depth-3 tree is readable and can represent simple interactions without an extensive parameter search. `min_samples_leaf=100` avoids tiny training leaves; seed 42 fixes randomness. These choices were frozen before June was accessed, rather than selected on final-test performance.

The five features are daily mean impressions, CTR percent, impression-weighted position, active-day share and daily impression CV, all from March 1–14. Nonpositive and sub-one daily positions are unknown; training-only median imputation handles missing position. This quality change follows the Week 4 review. No IDs, future counts, label proxies or product flags are features.

The label is a >20% drop in impressions between the equal-length March 1–14 and March 17–30 windows. It describes visibility movement, not editorial actionability or refresh benefit.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
from IPython.display import display, Markdown, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "skills/README.md").exists())
sys.path.insert(0, str(ROOT / "work/scripts"))
import capstone_pipeline as cp
SUMMARY_PATH = ROOT / "work/outputs/capstone_metrics.json"
if not SUMMARY_PATH.exists():
    cp.execute()
s = json.loads(SUMMARY_PATH.read_text())
print("Dataset revision:", s["dataset_revision"])
print("Protocol:", s["protocol_version"], s["protocol_sha256"])

raw, source_audit = cp.load_month("2026-03")
all_pages, X_all = cp.prepare(raw, "2026-03")
known = all_pages.label_known
pages = all_pages.loc[known].reset_index(drop=True)
X = X_all.loc[known].reset_index(drop=True)
y = pages.declined.astype(int)
display(pd.DataFrame([source_audit]))
display(X.head(5))
print(f"{len(pages):,} labeled pages; {X.shape[1]} past-only features.")

/var/home/tanzimul/Repos/github.com/tanzimul3islam/flyrank-ml-internship-starter/work/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset revision: 50cbf7c3909d07be4d1b5906b4d09e882e5acbf2
Protocol: capstone_v1 181f2c24f2742facb0f37fdf77f1ab5727a2b5cfa5876477fe235e63117e437f


,source_rows,first_date,last_date,clients,gsc_available_rows,gsc_false_rows,gsc_null_rows,subone_position_rows,duplicate_keys,identical_extra_rows_removed,conflicting_keys,clean_page_day_rows,partition,source_grain_violations_before_cleaning,source_grain_violations_after_cleaning,aggregate_rows
0,9841378,2026-03-01,2026-03-31,55,3611061,6230317,0,101548,0,0,0,9841378,fact_content_daily_performance/month=2026-03/d...,0,0,176738


,mean_daily_impressions,ctr_pct,impression_weighted_position,active_day_share,daily_impression_cv
0,8.142857,0.877193,12.421053,1.0,0.321105
1,19.428571,0.000000,12.466912,1.0,0.359618
2,28.500000,0.501253,11.706767,1.0,0.283430
3,112.714286,0.887199,7.308619,1.0,0.703812
4,92.571429,0.000000,56.003858,1.0,0.299108


57,624 labeled pages; 5 past-only features.


## 2. Split design

Use `GroupShuffleSplit(test_size=0.25, random_state=42)` by client, preserving the Week 4 cohort. Within the 24 development clients, five-fold GroupKFold provides an internal diagnostic. There is no grid search, refit on holdout clients or selection among many models. The March comparison has already been inspected; the separate June evaluation is documented in Week 6.

In [2]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
train, test = next(GroupShuffleSplit(n_splits=1, test_size=.25, random_state=42).split(X,y,groups=pages.client_hash_id))
assert set(pages.iloc[train].client_hash_id).isdisjoint(set(pages.iloc[test].client_hash_id))
baseline_receipt = json.loads((ROOT / "work/outputs/w04_baseline_metrics.json").read_text())
assert cp.fingerprint(pages.iloc[test]) == baseline_receipt["comparison_keys_sha256"]
assert cp.fingerprint(pages.iloc[train]) == baseline_receipt["development_keys_sha256"]
fold_results=[]
for fold,(a,b) in enumerate(GroupKFold(n_splits=5).split(X.iloc[train],y.iloc[train],pages.iloc[train].client_hash_id),1):
    tr,va=train[a],train[b]
    fitted=cp.make_model().fit(X.iloc[tr],y.iloc[tr])
    validation=pages.iloc[va].copy()
    validation["baseline_score"]=validation.past_impressions
    validation["model_score"]=fitted.predict_proba(X.iloc[va])[:,1]
    results=cp.comparison(validation)
    fold_results.append({"fold":fold,"validation_pages":len(va),"tree_auc":results["model"]["roc_auc"],
                         "tree_p20":results["model"]["precision_at_k"],"baseline_p20":results["baseline"]["precision_at_k"]})
display(pd.DataFrame(fold_results))
print(f"Train: {len(train):,} pages; comparison: {len(test):,}; no client overlap.")

,fold,validation_pages,tree_auc,tree_p20,baseline_p20
0,1,3233,0.520768,0.35,0.15
1,2,2745,0.560131,0.65,0.55
2,3,2721,0.531379,0.60,0.15
3,4,2719,0.582218,0.90,0.05
4,5,2719,0.509803,0.50,0.35


Train: 14,137 pages; comparison: 43,487; no client overlap.


## 3. Train and compare with the frozen baseline

The baseline score is past impressions; the tree's score is its uncalibrated positive-leaf frequency. Both use the same eligibility and the same held-out page IDs. Ties are deterministic using pseudonymous keys, which are never learned features. The primary task metric is precision@20, with prevalence shown as the expected value for a random queue. ROC-AUC and average precision assess more of the ranking.

In [3]:
model=cp.make_model().fit(X.iloc[train],y.iloc[train])
comparison=pages.iloc[test].copy()
comparison["baseline_score"]=comparison.past_impressions
comparison["model_score"]=model.predict_proba(X.iloc[test])[:,1]
results=cp.comparison(comparison)
display(pd.DataFrame(results).T)
assert results==s["march_comparison"]
print("Same split confirmed; March results match the aggregate capstone receipt.")

,n,clients,base_rate,k,top_k_declines,precision_at_k,roc_auc,average_precision,top_k_clients
baseline,43487.0,8.0,0.374365,20.0,9.0,0.45,0.532410,0.398072,3.0
model,43487.0,8.0,0.374365,20.0,14.0,0.70,0.603773,0.445687,2.0


Same split confirmed; March results match the aggregate capstone receipt.


## 4. Errors and interpretation

The tree improves March pooled precision@20 from 45% to 70%, but six of its first twenty do not decline. Its top twenty span only two clients, versus three for the baseline. These are proxy errors and concentration warnings; neither a positive nor a negative label establishes the correct editorial action.

Read the actual tree below. Its thresholds are associations learned from past measurements, not a reconstruction of Google's algorithm. June's permutation interpretation and adverse client-level results appear in the validation notebook and paper; no retraining follows them.

In [4]:
from sklearn.tree import export_text
print(export_text(model[-1],feature_names=cp.FEATURES,decimals=3))
top=cp.ordered(comparison,"model_score").head(20)
display(pd.DataFrame({"measure":["Top-20 declines","Top-20 non-declines","Top-20 clients"],
                     "value":[int(top.declined.sum()),int((top.declined==0).sum()),top.client_hash_id.nunique()]}))
assert model.n_features_in_==5
assert list(X.columns)==cp.FEATURES

|--- ctr_pct <= 0.361
|   |--- mean_daily_impressions <= 17.036
|   |   |--- mean_daily_impressions <= 12.179
|   |   |   |--- class: 0
|   |   |--- mean_daily_impressions >  12.179
|   |   |   |--- class: 0
|   |--- mean_daily_impressions >  17.036
|   |   |--- impression_weighted_position <= 26.338
|   |   |   |--- class: 0
|   |   |--- impression_weighted_position >  26.338
|   |   |   |--- class: 1
|--- ctr_pct >  0.361
|   |--- ctr_pct <= 0.376
|   |   |--- daily_impression_cv <= 0.544
|   |   |   |--- class: 0
|   |   |--- daily_impression_cv >  0.544
|   |   |   |--- class: 0
|   |--- ctr_pct >  0.376
|   |   |--- daily_impression_cv <= 0.370
|   |   |   |--- class: 0
|   |   |--- daily_impression_cv >  0.370
|   |   |   |--- class: 0



,measure,value
0,Top-20 declines,14
1,Top-20 non-declines,6
2,Top-20 clients,2


## 5. Self-check

- [x] One fixed model, training-only imputation, no parameter search.
- [x] Five past-only features and a grouped split verified against Week 4 fingerprints.
- [x] Baseline, model and prevalence compared on identical pages.
- [x] Errors and client concentration inspected; no causal or editorial-success claims.
- [x] Notebook executed with outputs; code and aggregate receipts support reproduction.

AI assistance drafted and executed this notebook. The final June result is evaluated separately, not used to retune this model.